# Kernel Leak Lab — Beginner
You are evaluating **WPInference**, an AI accelerator that applies a convolution kernel to image data. Its input values are known, but its model weights are confidential. Your task is to recover three weights using power traces recorded during its calculations.

A **weight** is a learned numerical parameter. A **kernel** is a small grid of weights used to combine neighboring image values. Here, one 3×3 kernel contains nine weights. Each weight multiplies one input value, and the products are added together. You will investigate one such calculation in the first convolution layer, where the inputs are directly known.

A **side channel** is an observation of how a computation runs that can reveal information about the values being processed. In this lab, the observation is simulated power consumption. You will predict power-dependent behavior for possible weights, compare those predictions with measurements, and decide which weights the evidence supports.

Run cells in order. Edit the input choices, weight guesses, and evidence notes as you progress. Keep one recorded run for your submission.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lab import Investigation, hw32, ranking, plot_trace, plot_candidates

# Load the challenge metadata and start a new acquisition ledger.
# Re-running this setup cell resets the investigation, so run it once per session.
lab = Investigation()
print("Instance:", lab.manifest["instance_id"])
print("Known first weight:", lab.known)
print("Target weight indices: 1, 2, 3. All required weights are nonzero.")

The **instance ID** identifies your assigned dataset so the instructor can match your submission to its answer key. The known first weight initializes the prefix for challenge recovery. Target indices use zero-based indexing: weight 0 is known; weights 1, 2, and 3 are the next three MAC operands.

## 1 · Inputs, weights, and the attack model
The **attack model** specifies what the evaluator knows and can observe. You know the input values, the kernel size, the order of operations, the sample positions associated with those operations, and the first weight. You can request traces for selected input patterns. You cannot read the model weights from memory.

WPInference performs nine **multiply-accumulate (MAC)** operations for the selected 3×3 input window. A MAC multiplies an input by a weight and adds that product to a running total called the **accumulator**. We label the weights 0 through 8, reading the kernel row by row. The required targets are weights 1, 2, and 3; weight 0 is provided.

For example, if the accumulator contains 2, the next input is 3, and the next weight is 5, the new accumulator value is `2 + 3*5 = 17`. Although the final sum is one number, the accumulator passes through a different intermediate state after each MAC. Those intermediate states are the target of the attack.

### Representation and arithmetic
Inputs and weights use **signed int8** values: integers from −128 to 127 stored in eight bits. **Quantization** represents neural-network values using a limited set of integers. A zero point specifies which stored integer represents numerical zero; this lab uses zero point 0. Input packets already contain these quantized values.

Products are computed after widening the operands, and sums are held in a **signed 32-bit accumulator**. It starts at zero for each execution and retains its running sum between MACs. Bias, an additional offset sometimes used in a neural-network layer, is zero here. There is no rounding or saturation inside the nine-MAC calculation, and all the sums fit within the accumulator's range.

For execution `r` and target weight index `j`:

```text
Aold[r] = sum(x[r,k] * weight[k] for k < j)
Anew[r] = Aold[r] + x[r,j] * candidate_weight
```

`x[r,k]` means input position `k` in execution `r`. `Aold` and `Anew` are the accumulator states before and after the target MAC. To predict them, you need the known inputs and the weights preceding the target. This is why recovery proceeds in order.

**Before inspecting the traces:** predict whether encrypting a stored model file would prevent this attack. Consider what happens when the accelerator uses the weights during inference.

## 2 · Predicting register switching
A register stores a number as a pattern of bits. Updating the accumulator changes some of those bits. The lab models the data-dependent part of power consumption using the number of bit transitions during the update.

**Hamming weight (HW)** counts the set bits in a value. For example, 7 is `111` in binary, so HW(7) is 3.

**Hamming distance (HD)** counts bit positions that differ between two states. XOR, written `^` in Python, produces a 1 wherever two bits differ, so `HD(old,new) = HW(old XOR new)`. For the transition 2 → 7, compare `010` with `111`: two positions change, so HD is 2.

Before running the cell, calculate HW(7), HD(2,7), and the 32-bit HD for −1 → 0. Negative values use **two's complement** representation. In 32 bits, −1 is all ones and zero is all zeros. The helper `hw32` encodes a value modulo 2³² before counting bits, so negative numbers are handled as fixed-width register states.

In [ ]:
# Count set bits directly, then use XOR to count changed bit positions.
print("HW(7) =", int(hw32(7)))
print("HD(2,7) =", int(hw32(2 ^ 7)))
print("32-bit HD(-1,0) =", int(hw32(-1 ^ 0)))

# Load the guided inputs and corresponding power observations.
# Matching row numbers identify the same execution in both arrays.
with np.load("packets/guided.npz", allow_pickle=False) as packet:
    gx, gt = packet["inputs"].copy(), packet["traces"].copy()
print("Guided inputs (executions, input positions):", gx.shape)
print("Guided traces (executions, power samples):", gt.shape)

# Plot the first execution, with guide lines at the nine MAC centers.
plot_trace(gt)
plt.show()

### Reading a power trace
A **trace** is a sequence of power samples collected during one execution. The horizontal axis of the plot above is the sample index; the vertical axis is simulated power in arbitrary units. A sample index identifies a position in the trace; it is not a duration in seconds.

The three printed bit-count results should be **3, 2, and 32**. The trace plot contains nine MAC updates. Vertical guide lines identify their expected sample centers. Noise makes the curve irregular, so one trace does not directly reveal a weight.

The loaded arrays are `gx` (guided inputs) and `gt` (guided traces). Each row represents one execution. `gx[r,j]` is input `j` for execution `r`; `gt[r,t]` is power sample `t` from that same execution. There are 128 guided executions, nine input values per execution, and 108 power samples per trace.

The guided example is a separate kernel with known first weight **37**. MAC index `j` has sample center `6 + 12*j`, so the first MAC is observed at sample 6. Starting from zero, its switching prediction is the HW of `gx[:,0]*37`. In array notation, `:` selects every row: the prediction contains one value per execution.

In [ ]:
# Widen the input integers before multiplying to avoid int8 overflow.
# The first MAC starts at zero, so its HD equals the product's HW.
known_prediction = hw32(gx[:, 0].astype(np.int64) * 37)

# Compare one prediction and one measured sample per execution.
# corrcoef returns a 2x2 matrix; [0,1] is the correlation between the vectors.
calibration_r = np.corrcoef(known_prediction, gt[:, 6])[0, 1]
print("Calibration correlation:", calibration_r)

**Read the output:** the calibration coefficient compares predicted switching counts with the measured first-MAC power across all 128 guided executions. A strong positive result supports using positive correlation for subsequent rankings. Noise keeps it below a perfect +1. If the sign were reversed, you would need to revisit the measurement polarity before ranking candidates.

## 3 · Correlation power analysis
**Correlation power analysis (CPA)** compares predicted leakage with measured power across many executions. For each possible weight, it calculates the intermediate accumulator update, converts the update to a predicted bit count, and compares that prediction vector with the measured power vector.

The comparison uses **Pearson's correlation coefficient**, `r`, which ranges from −1 to +1. A value near +1 means larger predicted counts tend to accompany larger measured samples; a value near −1 means the relationship is reversed. A value near zero indicates little linear relationship. The calibration cell should give a strong positive value because this dataset uses positive gain: more modeled transitions contribute more power.

The coefficient is not a probability that a weight is correct. A wrong candidate can resemble the correct candidate, and noise can change their order. You will compare the leading candidate with the **runner-up**, the next-highest candidate, and later test your selection on fresh observations.

The next cell targets guided weight index 1. It compares two hypotheses: **HD**, which models transitions from the previous accumulator state, and **HW**, which models set bits in the new state alone. Both use the known preceding weight `[37]`. Predict which will fit the stated register behavior better.

In [ ]:
# Analyze the same target and measurements under each leakage model.
for model in ("HD", "HW"):
    # [37] supplies the known weight preceding target index 1.
    # [:3] displays the three highest-correlating candidates.
    print(model, ranking(gx, gt, [37], 1, model)[:3])
    plot_candidates(gx, gt, [37], 1, model)
plt.show()

### Explaining the guided result
Each printed dictionary contains a candidate `weight` and its correlation `r`. The list is ordered from highest to lowest positive correlation. The plots show how correlation varies across signed int8 candidates. Compare the HD and HW results: which has a more convincing leading candidate, and does that match the register model?

The **runner-up gap** is the leading correlation minus the second-highest correlation. A small gap means the two predictions fit similarly. A large absolute correlation alone is insufficient if many neighboring candidates fit almost as well. Record the leading weight, runner-up, their correlations, and their gap. Explain how a known input and proposed weight become an intermediate state and then a leakage prediction.

### When correlation cannot identify a weight
If every execution uses the same input, each candidate produces the same prediction on every row. Pearson correlation is undefined for a constant prediction because its standard deviation is zero. Repeating the same input can help average measurement noise, but a CPA experiment still needs variation across distinct inputs. With several distinct input groups, repeats can be averaged within each group.

Different weights can also produce identical leakage predictions. With a zero old state and nonnegative inputs from 0 to 127, weights 1 and 2 give products with the same HW: multiplying by two shifts the positive bit pattern left without changing the number of ones. More traces of that experiment cannot distinguish them. The required challenge has been checked for exact correlation aliases along its diverse-input route.

## 4 · Choosing your measurements
Return to WPInference's challenge kernel. Its known first weight is `lab.known`; use that value rather than assuming it is 37.

You have a budget of **320 raw executions**. Use up to 256 traces to discover the weights and reserve 64 fresh traces for validation. Each execution contains observations for all three targets, so you can reuse the same discovery traces when moving to the next weight.

Choose batches of 32, 64, or 128:

- **`diverse`** spans the signed input range, producing a broad variety of accumulator updates.
- **`narrow`** uses input values from 0 through 3, offering less variation.
- **`repeated`** measures the same input repeatedly, offering a way to examine noise without changing the predicted computation.

Before acquiring a batch, write what you expect that choice to help you determine. The prediction in the cell is a starting example; edit it to reflect your decision. Re-running the cell collects the next batch and spends additional credits. Start with 32 and inspect the results before deciding whether to acquire more.

In [ ]:
# Choose the input strategy, batch size, and your prediction before collecting.
# A re-run adds another batch; it does not replace the previous acquisition.
acquisition = lab.acquire(
    "diverse", 32,
    "Different signed inputs should provide variation in the switching predictions."
)
print(acquisition)

**Read the output:** `acquired` is the size of this batch, `discovery_used` is the cumulative trace count, and `credits_left_including_validation` includes the 64 executions reserved for validation. After the initial batch of 32, those values should be 32, 32, and 288. The next analysis cell uses all discovery batches collected so far.

### Recovering the weights in sequence
The next cell uses a **prefix**, a list of weights preceding the target. Initially, `previous_guesses = []` is empty, so the prefix contains only the provided weight and the cell analyzes target 1.

After choosing weight 1, enter it in `previous_guesses`, for example `[your_first_guess]`, and re-run the analysis cell to examine target 2. After choosing weight 2, enter both guesses in order to examine target 3. Keep the first value in the prefix as `lab.known`. Use integer values you selected from your own results.

The previous accumulator state is computed separately for every execution. An incorrect earlier guess changes those states and can make the next weight difficult to recover. If later results are unconvincing, revisit the preceding estimate and its evidence.

A candidate weight of zero always predicts zero HD: adding an input multiplied by zero leaves the accumulator unchanged. Its constant predictions give undefined correlation, so it is excluded from the ranking. All required target weights are specified to be nonzero.

In [ ]:
# Enter zero, one, or two selected weights preceding the next target.
previous_guesses = []

# The known first weight always begins the prefix.
prefix = [lab.known] + previous_guesses
print("Target index:", len(prefix))

# Rank candidates using every discovery trace acquired so far.
print(lab.rank(prefix))
plot_candidates(lab.inputs, lab.traces, prefix, len(prefix))
plt.show()

# Review which measurements contributed to these results.
print("Acquisition ledger:", lab.ledger)

### Interpreting your candidate output
`Target index` identifies the weight currently being analyzed. The printed list contains the ten highest-correlating candidates. The plot shows all candidates with defined correlations; the missing point at zero corresponds to the constant zero-weight prediction. The ledger lists each acquisition, your prediction, and the cumulative number of discovery traces used.

For each target, record your chosen weight, the runner-up, both correlations, their gap, the leakage model, the sample center, and the trace count. If a candidate barely leads, consider whether another batch of inputs would help. Compare rankings before and after acquisition rather than assuming more traces must confirm your first guess.

If you reach the budget with inconclusive results, explain what you observed and which measurement choice you would change. Ask your instructor about a rescue run while retaining the failed-run evidence.

## 5 · Independent validation
Once you have selected weights 1, 2, and 3, enter them in `guesses` in that order. **Freezing** records that tuple and closes discovery acquisition for the current investigation.

Validation evaluates the selected tuple on 64 fresh diverse-input executions. These observations were not used to choose the candidates. This checks whether the apparent fit persists beyond the discovery data. Leave the list empty until you are ready.

If you change a guess after inspecting validation, that batch has influenced your selection. A revised tuple needs a new validation batch; ask the instructor how to proceed.

In [ ]:
# Enter your three selected weights as Python integers, in target order.
guesses = []
if len(guesses) == 3:
    # Freeze once, before accessing the independent validation observations.
    if lab.frozen is None:
        lab.freeze(guesses)
    if guesses != lab.frozen:
        print("Your edited guesses differ from the frozen tuple; this run still validates", lab.frozen)
    # Re-running returns the same validation result, not another fresh batch.
    print(lab.validate())
else:
    print("Select three weights before freezing and opening validation.")

**Read the output:** each validation entry identifies the target `index`, your `chosen` weight, its `held_out_r`, and its `rank` among candidates on the fresh observations. Rank 1 means no candidate had a higher positive correlation. A larger rank indicates that another candidate fit better. `None` means the correlation was undefined. Validate the whole tuple carefully: later targets still depend on the earlier selected weights. These ranks provide a consistency check; the instructor's exact check establishes which weights match.

## 6 · Comparing defenses
Before opening the results, predict the effect of each change:

1. **Storage encryption:** encrypt the model file, then use the same weights and MAC arithmetic during inference.
2. **More independent noise:** add random variation to each measured sample.

The baseline provides the comparison point. The storage-encryption packet uses the same observations as the baseline because it represents unchanged accumulator switching during inference. The noise packet increases noise standard deviation from 5 to 10 arbitrary units. Standard deviation measures the typical spread of the added random variation.

All variants use the same inputs so the comparison focuses on the defense condition. Open these packets after freezing your selections. They are comparison data, outside the discovery and validation budget.

In [ ]:
# Open the defense comparison only after recording your weight selections.
if lab.frozen is not None:
    for name in ("baseline", "storage_encryption", "more_noise"):
        with np.load(f"packets/defense_{name}.npz", allow_pickle=False) as p:
            # Use the same first target and known prefix for each condition.
            rows = ranking(p["inputs"], p["traces"], [lab.known], 1)
            print(name, rows[:2])  # Display the leader and runner-up.
else:
    print("Write predictions now; open defense observations after freezing.")

### Interpreting the defense results
Each line reports a defense condition and its two leading candidates for weight 1. Compare the winning correlation and runner-up gap with the baseline. The unchanged observations for storage encryption should give the same ranking as the baseline. The extra-noise results may make a candidate less convincing, but one finite experiment does not establish that extraction is impossible.

Storage encryption protects the model file. During ordinary inference, the MAC still processes the weights, so the modeled register transitions remain observable.

Independent noise can be reduced statistically by collecting more measurements. Another defense direction is **masking**, which splits sensitive values into randomized shares so a single observed intermediate need not directly reveal the original value. Designing the arithmetic and preventing unintended leakage between shares require care. Explain one proposed defense, its expected effect on this attack, and a cost or remaining verification need.

## 7 · Evidence submission and transfer question
Complete the six notes below using values from your own run:

- **`hd_calculation`:** show one accumulator update and explain its fixed-width bit count.
- **`acquisition_observation`:** compare an acquisition prediction with what you observed.
- **`candidate_evidence`:** give your candidate/runner-up results and interpret the fresh validation.
- **`defense`:** explain the comparison and propose a mitigation with a tradeoff.
- **`threat_model`:** state what the evaluator knows and observes, plus one assumption that limits the conclusion.
- **`transfer_answer`:** answer: **A teammate recovers weight 1 incorrectly but uses the true inputs for weight 2. Why might weight 2 fail, and what experiment would you do next?**

Use AI assistance according to course rules and disclose it as required. Ground your explanations in your recorded observations.

In [ ]:
# Replace each empty string with your written evidence.
lab.notes = {
    "hd_calculation": "",
    "acquisition_observation": "",
    "candidate_evidence": "",
    "defense": "",
    "threat_model": "",
    "transfer_answer": "",
}

# Export only after validation and all six written responses are complete.
if lab.validation is not None and all(lab.notes.values()):
    print(lab.export("submission.json"))
else:
    print("Complete validation and all six evidence notes, then re-run to export.")

Submit `submission.json` and your completed notebook. A successful export prints the file's absolute path; the file contains your selected weights, acquisition ledger, validation results, and written evidence. Your instructor checks exact weight matches separately.

The result concerns three weights in an untrained first-layer kernel, modeled as a sequential MAC lane. Recovering those weights does not establish the behavior of a complete classifier. Parallel accelerators mix activity from multiple units, so extending the attack would require revisiting the leakage model and the observations available to the evaluator.